# Experiment 8.1 — Hidden quantization ablation

Analysis-only notebook for the finalized Exp8.1 artifacts. The experiment compares binary communication, cap-31 weighted multi-spike communication, and fixed heterogeneous-threshold binary populations under the Exp7.3 A2 training contract.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ART = ROOT / "notebooks" / "artifacts" / "experiment_8_1_hidden_quantization_ablation" / "hidden_quantization_ablation_v1"

manifest = json.loads((ART / "manifest.json").read_text())
runs = pd.read_csv(ART / "method_runs.csv")
summary = pd.read_csv(ART / "method_summary.csv")
contrasts = pd.read_csv(ART / "contrast_summary.csv")
activity = pd.read_csv(ART / "activity_summary.csv")
manifest


## Method-level results

The primary endpoint is native Linear/WCCE test balanced accuracy. L1/L2 Fixed250 probes show whether a communication code preserves local temporal information, while the quantization gap compares pre-reset membrane information with the value actually transmitted to the next layer.


In [ ]:
cols = [
    "method", "linear_test_ba_mean", "linear_test_ba_std",
    "l1_pre_reset_fixed250_ba_mean", "l1_communication_fixed250_ba_mean",
    "l1_quantization_gap_fixed250_mean",
    "l2_pre_reset_fixed250_ba_mean", "l2_communication_fixed250_ba_mean",
]
summary[[c for c in cols if c in summary.columns]]


In [ ]:
order = manifest["method_order"]
plot_df = runs.groupby("method", sort=False)["linear_test_ba"].agg(["mean", "std"]).reindex(order)
ax = plot_df["mean"].plot(kind="bar", yerr=plot_df["std"], capsize=4, figsize=(9, 4))
ax.set_ylabel("Test balanced accuracy")
ax.set_xlabel("Method")
ax.set_title("Exp8.1 native Linear/WCCE performance")
plt.tight_layout()
plt.show()


## Quantization-gap view


In [ ]:
gap = runs.groupby("method", sort=False)[[
    "l1_pre_reset_fixed250_ba",
    "l1_communication_fixed250_ba",
    "l1_quantization_gap_fixed250",
]].mean().reindex(manifest["method_order"])
gap


In [ ]:
ax = gap[["l1_pre_reset_fixed250_ba", "l1_communication_fixed250_ba"]].plot(kind="bar", figsize=(10, 4))
ax.set_ylabel("L1 Fixed250 probe BA")
ax.set_xlabel("Method")
ax.set_title("Pre-reset state vs communicated L1 representation")
plt.tight_layout()
plt.show()


## Paired contrasts

These contrasts separate event-resolution, threshold-heterogeneity, and pure-width effects.


In [ ]:
contrast_cols = [c for c in contrasts.columns if c in {"contrast", "method_a", "method_b"} or c.startswith("delta_linear_test_ba") or c.startswith("delta_l1_communication_fixed250_ba") or c.startswith("delta_l1_quantization_gap_fixed250")]
contrasts[contrast_cols]


## Activity by threshold/tau group


In [ ]:
activity.head(30)
